# 04 · Results dashboard (Phase 11)

Companion to `docs/final_report.md` and the live dashboard at
<https://blessblissmari.github.io/flybrain-results/>.

This notebook:
1. Loads `comparison_overall.json` from a `flybrain-py bench` output
   directory.
2. Renders the headline scoreboard, the cost-vs-success scatter, and the
   per-benchmark drill-down inline.
3. Surfaces the same cherry-picked traces used in `data/traces/sample/`
   so a reviewer can step through one solved + one failed trace without
   leaving the notebook.

Inputs are the JSON artefacts emitted by Phase 10's CLI; nothing here
depends on a network or a YandexGPT key. Point `BENCH_DIR` at any
benchmark output (e.g. `runs/bench_yandex` or the bundled smoke run).

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BENCH_DIR = Path('runs/bench_yandex')
if not BENCH_DIR.exists():
    BENCH_DIR = Path('/tmp/bench_yandex')
BENCH_DIR.resolve()

## Headline comparison

Loads `comparison_overall.json`, resolves strict-JSON `null` ⇒ `inf`
for `cost_per_solved_rub`, sorts by success.

In [ ]:
rows = json.loads((BENCH_DIR / 'comparison_overall.json').read_text())
for r in rows:
    if r.get('cost_per_solved_rub') is None:
        r['cost_per_solved_rub'] = math.inf
df = pd.DataFrame(rows).sort_values('success_rate', ascending=False)
df[['name', 'success_rate', 'verifier_pass_rate',
    'avg_total_tokens', 'avg_llm_calls', 'avg_cost_rub',
    'cost_per_solved_rub']]

## Cost vs success scatter

The same plot as the dashboard's overview tab, rendered with matplotlib.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for _, r in df.iterrows():
    if not math.isfinite(r.avg_cost_rub):
        continue
    ax.scatter(r.avg_cost_rub, r.success_rate, s=140,
               edgecolor='white', linewidth=1.0)
    ax.annotate(r['name'], (r.avg_cost_rub, r.success_rate),
                xytext=(8, 0), textcoords='offset points', fontsize=9)
ax.set_xlabel('avg cost / task (RUB)')
ax.set_ylabel('success rate')
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.2)
ax.set_title('Cost vs success — full_min')
plt.tight_layout()
plt.show()

## Per-benchmark breakdown

Loads each `comparison_<bench>.json` and stacks success rates per
baseline. This is the same view as the dashboard's per-benchmark tab.

In [ ]:
per_bench = {}
for f in sorted(BENCH_DIR.glob('comparison_*.json')):
    if f.name == 'comparison_overall.json':
        continue
    bench = f.stem.replace('comparison_', '')
    per_bench[bench] = pd.DataFrame(json.loads(f.read_text()))
stack = pd.concat([
    df.assign(benchmark=bench)[['name', 'benchmark', 'success_rate']]
    for bench, df in per_bench.items()
])
pivot = stack.pivot(index='name', columns='benchmark', values='success_rate')
pivot.style.format('{:.2f}').background_gradient(cmap='Greens', axis=None, vmin=0, vmax=1)

## Cherry-picked traces

Print the same two traces that ship under `data/traces/sample/`. The
headers tell you what to look for; the JSON below is the full Phase-10
trace schema (steps, totals, verification).

In [ ]:
from textwrap import indent

samples = Path('data/traces/sample')
if not samples.exists():
    samples = Path('../data/traces/sample')
for f in sorted(samples.rglob('*.trace.json')):
    trace = json.loads(f.read_text())
    v = trace.get('verification') or {}
    print(f"{f.relative_to(samples)}")
    print(f"  passed={v.get('passed')!r}  score={v.get('score')!r}  failed_component={v.get('failed_component')!r}")
    print(indent('steps:', '  '))
    for i, step in enumerate(trace.get('steps') or [], 1):
        action = step.get('graph_action') or {}
        print(f"    {i:>2}. {action.get('kind', 'unknown'):<14}  {action.get('agent', '')}")
    print()